# Tabular Safe RL Baselines

This notebook compares MASA's tabular safe RL baselines on the same constrained `ColourGridWorld` setup. The runs are intentionally tiny: the goal is to understand the safety mechanism each algorithm adds, not to produce benchmark rankings.

The matching static docs page is at `docs/Tutorials/Baselines/Tabular Safe RL Baselines.md`.

## CPU-first setup

Set CPU-oriented environment variables before importing MASA/JAX modules (for portability).

In [1]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {0: "left", 1: "right", 2: "down", 3: "up", 4: "stay"}

## Imports

The five classes below correspond to the registered algorithm names `QL`, `QL_LAMBDA`, `LCRL`, `SEM`, and `RECREG`.

In [ ]:
from IPython.display import Markdown, display

import masa
from masa.common.utils import make_env
from masa.envs.tabular.colour_grid_world import BLUE_STATE, GOAL_STATE, cost_fn, label_fn
from masa.algorithms.tabular import LCRL, QL, QL_LAMBDA, RECREG, SEM
from notebooks.tutorials.helpers.tabular_safe_rl_baselines import (
    algorithm_state_summary,
    markdown_table,
    smoke_run_note,
)


I0000 00:00:1779206160.300815  105019 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779206162.798601  105019 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Shared environment

All algorithms see the same CMDP environment. The task reward is sparse and positive at `goal`; safety cost is positive at the blue state. This is a small enough setting for notebook execution but still separates reward and safety.

In [3]:
ENV_ID = "ColourGridWorld"
CONSTRAINT = "CMDP"
MAX_EPISODE_STEPS = 40
BUDGET = 0.0

def make_train_env():
    return make_env(
        ENV_ID,
        CONSTRAINT,
        MAX_EPISODE_STEPS,
        label_fn=label_fn,
        constraint_kwargs={"cost_fn": cost_fn, "budget": BUDGET}
    )


def make_eval_env():
    return make_env(
        "ColourGridWorld",
        "CMDP",
        40,
        label_fn=label_fn,
        constraint_kwargs={"cost_fn": cost_fn, "budget": 0.0}
    )

print("unsafe blue state:", BLUE_STATE)
print("rewarding goal state:", GOAL_STATE)

unsafe blue state: 36
rewarding goal state: 80


## Probe the reward/safety signals first

Before training, run two deterministic scripts. The unsafe script reaches `blue`; the goal script reaches `goal` without visiting `blue`. This gives the metrics that every algorithm will later receive through `info`.

In [4]:
def run_script(seed, actions):
    env = make_train_env()
    obs, info = env.reset(seed=seed)
    rows = [
        {
            "event": "reset",
            "obs": int(obs),
            "labels": sorted(info["labels"]),
            "constraint_step": info["constraint"]["step"],
        }
    ]
    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        rows.append(
            {
                "event": f"step_{step}",
                "action": ACTION_NAMES[action],
                "obs": int(obs),
                "reward": float(reward),
                "terminated": bool(terminated),
                "truncated": bool(truncated),
                "labels": sorted(info["labels"]),
                "constraint_step": info["constraint"]["step"],
                "constraint_episode": info["constraint"].get("episode", {}),
                "metrics_episode": info.get("metrics", {}).get("episode", {}),
            }
        )
        if terminated or truncated:
            break
    env.close()
    return rows

unsafe_rows = run_script(seed=1, actions=[2, 2, 2, 2])
goal_rows = run_script(seed=4, actions=[2] * 8 + [1] * 8)

print("Unsafe script final row:")
pprint(unsafe_rows[-1])
print("\nGoal script final row:")
pprint(goal_rows[-1])

assert unsafe_rows[-1]["constraint_step"]["violation"] == 1.0
assert unsafe_rows[-1]["constraint_episode"]["satisfied"] == 0.0
assert goal_rows[-1]["reward"] == 1.0
assert goal_rows[-1]["constraint_episode"]["satisfied"] == 1.0

Unsafe script final row:
{'action': 'down',
 'constraint_episode': {'cum_cost': 1.0, 'satisfied': 0.0},
 'constraint_step': {'cost': 1.0, 'cum_cost': 1.0, 'violation': 1.0},
 'event': 'step_4',
 'labels': ['blue'],
 'metrics_episode': {},
 'obs': 36,
 'reward': 0.0,
 'terminated': False,
 'truncated': False}

Goal script final row:
{'action': 'right',
 'constraint_episode': {'cum_cost': 0.0, 'satisfied': 1.0},
 'constraint_step': {'cost': 0.0, 'cum_cost': 0.0, 'violation': 0.0},
 'event': 'step_16',
 'labels': ['goal'],
 'metrics_episode': {'ep_length': 16, 'ep_reward': 1.0},
 'obs': 80,
 'reward': 1.0,
 'terminated': True,
 'truncated': False}


## Algorithm registry

Each entry uses the same environment factory and seed pattern. The configuration is intentionally small and explicit so it is easy to compare what changes between algorithms.

In [5]:
ALGORITHMS = {
    "QL": {
        "class": QL,
        "kwargs": {},
        "mechanism": "Task Q-table only; costs are logged but not penalized in the update target.",
    },
    "QL_LAMBDA": {
        "class": QL_LAMBDA,
        "kwargs": {"cost_lambda": 1.0},
        "mechanism": "Subtracts cost_lambda * cost from the reward target.",
    },
    "LCRL": {
        "class": LCRL,
        "kwargs": {"r_min": -1.0},
        "mechanism": "Maps violations to an absorbing-style value based on r_min / (1 - gamma).",
    },
    "SEM": {
        "class": SEM,
        "kwargs": {"r_min": -1.0, "cost_coef": 1.0},
        "mechanism": "Learns task Q plus auxiliary D and C safety tables that alter action selection.",
    },
    "RECREG": {
        "class": RECREG,
        "kwargs": {
            "mode": "model_free",
            "model_checking": "none",
            "horizon": 3,
            "safety_prob": 0.2,
            "cost_coef": 2.0,
        },
        "mechanism": "Learns task Q, backup B, risk S, and can report override_rate when actions are replaced.",
    },
}

display(Markdown(markdown_table(
    [
        {"algorithm": name, "mechanism": spec["mechanism"]}
        for name, spec in ALGORITHMS.items()
    ],
    ["algorithm", "mechanism"],
)))

| algorithm | mechanism |
| --- | --- |
| QL | Task Q-table only; costs are logged but not penalized in the update target. |
| QL_LAMBDA | Subtracts cost_lambda * cost from the reward target. |
| LCRL | Maps violations to an absorbing-style value based on r_min / (1 - gamma). |
| SEM | Learns task Q plus auxiliary D and C safety tables that alter action selection. |
| RECREG | Learns task Q, backup B, risk S, and can report override_rate when actions are replaced. |

## Tiny smoke training runs

These calls intentionally use `num_frames=20`, `eval_freq=10`, `log_freq=10`, and `num_eval_episodes=1`. Treat the printed tables as wiring checks and diagnostics, not as claims about which method is best.

In [6]:
def train_one(name, spec, seed=0):
    env = make_train_env()
    eval_env = make_eval_env()
    algo = spec["class"](
        env,
        eval_env=eval_env,
        seed=seed,
        device="cpu",
        verbose=0,
        exploration="epsilon_greedy",
        initial_epsilon=1.0,
        final_epsilon=0.2,
        epsilon_decay_frames=20,
        **spec["kwargs"],
    )
    algo.train(
        num_frames=20,
        eval_freq=10,
        log_freq=10,
        num_eval_episodes=1,
        stats_window_size=5,
    )
    env.close()
    eval_env.close()
    return algo

trained_algorithms = {}
for offset, (name, spec) in enumerate(ALGORITHMS.items()):
    print(f"\n=== {name} ===")
    trained_algorithms[name] = train_one(name, spec, seed=offset)

print("\n" + smoke_run_note())


=== QL ===


frames:   0%|                                                                                    | 0/20 [00:00…

evaluation:   0%|                                                                                 | 0/1 [00:00…

-----------------------------------
|  run/               |           |
|    fps              |  461.7    |
|    time_elapsed     |  0.02166  |
|    total_timesteps  |  10       |
-----------------------------------
|  train/rollout/     |           |
|    cum_cost         |  0        |
|    satisfied        |  1        |
-----------------------------------
|  train/stats/       |           |
|    alpha            |  0.1      |
|    epsilon          |  0.68     |
-----------------------------------



evaluation:   0%|                                                                                 | 0/1 [00:00…

...

frames:   0%|                                                                                    | 0/20 [00:00…


evaluation:   0%|                                                                                 | 0/1 [00:00<?, ?it/s]
                                                                                                                        

-----------------------------------
|  run/               |           |
|    fps              |  479.9    |
|    time_elapsed     |  0.02084  |
|    total_timesteps  |  10       |
-----------------------------------
|  train/rollout/     |           |
|    cum_cost         |  0        |
|    satisfied        |  1        |
-----------------------------------
|  train/stats/       |           |
|    alpha            |  0.1      |
|    safe_alpha       |  0.1      |
|    epsilon          |  0.68     |
-----------------------------------




evaluation:   0%|                                                                                 | 0/1 [00:00<?, ?it/s]
                                                                                                                        

-----------------------------------
|  run/               |           |
|    fps              |  445.8    |
|    time_elapsed     |  0.04487  |
|    total_timesteps  |  20       |
-----------------------------------
|  train/rollout/     |           |
|    cum_cost         |  0        |
|    satisfied        |  1        |
-----------------------------------
|  train/stats/       |           |
|    alpha            |  0.1      |
|    safe_alpha       |  0.1      |
|    epsilon          |  0.28     |
-----------------------------------
|  eval/rollout/      |           |
|    cum_cost         |  1        |
|    satisfied        |  0        |
|    override_rate    |  0        |
|    ep_reward        |  0        |
|    ep_length        |  40       |
-----------------------------------


These are smoke-run diagnostics from tiny training runs. They prove that the algorithms, environments, and logs connect; they are not benchmark rankings.


## Inspect learned state

Even after a tiny run, the shape of each learned object shows what the method tracks. `QL`, `QL_Lambda`, and `LCRL` have a task `Q` table; `SEM` adds `D` and `C`; `RECREG` adds backup `B` and risk `S`.

In [7]:
summary_rows = [
    algorithm_state_summary(name, algo)
    for name, algo in trained_algorithms.items()
]

display(Markdown(markdown_table(
    summary_rows,
    ["algorithm", "Q shape", "max Q", "min Q", "D shape", "C shape", "B shape", "S shape"],
)))

assert trained_algorithms["QL"].Q.shape == (81, 5)
assert trained_algorithms["QL_LAMBDA"].cost_lambda == 1.0
assert hasattr(trained_algorithms["SEM"], "D") and hasattr(trained_algorithms["SEM"], "C")
assert hasattr(trained_algorithms["RECREG"], "B") and hasattr(trained_algorithms["RECREG"], "S")

| algorithm | Q shape | max Q | min Q | D shape | C shape | B shape | S shape |
| --- | --- | --- | --- | --- | --- | --- | --- |
| QL | (81, 5) | 0 | 0 |  |  |  |  |
| QL_LAMBDA | (81, 5) | 0 | 0 |  |  |  |  |
| LCRL | (81, 5) | 0 | 0 |  |  |  |  |
| SEM | (81, 5) | 1.36 | 0 | (81, 5) | (81, 5) |  |  |
| RECREG | (81, 5) | 0 | 0 |  |  | (81, 5) | (3, 81, 5) |

## Reading the tradeoff

- `QL` is the task-only reference point. It is useful precisely because it does not encode a safety preference in its target.
- `QL_LAMBDA` is the simplest penalty baseline: unsafe labels reduce the scalar return.
- `LCRL` treats violations more like entering a bad absorbing outcome than paying a small cost.
- `SEM` keeps task and safety estimates separate, then combines them during action selection.
- `RECREG` is intervention-oriented: it estimates risk, learns a backup policy, and can log `override_rate` when it replaces risky task actions.

For real comparisons, increase frames, run multiple seeds, record confidence intervals, and keep this notebook as a short correctness and interpretation check.